# Practice 5: Exploring Harmony in Pop Music

**Digital Musicology — The McGill Billboard Dataset**

---

In this session we work with the **McGill Billboard Dataset**, a corpus of expert chord annotations for songs sampled from the *Billboard* Hot 100 chart between 1958 and 1991. The dataset was created at McGill University and released under a CC0 licence (Burgoyne, Wild & Fujinaga, 2011).

We will use this dataset to explore the **harmonic language** of Billboard hits — which chords dominate, how they follow each other, and what this tells us about the conventions of popular music.

**What you will do today**

1. **Load** the McGill Billboard dataset (metadata + chord annotations)
2. **Explore** the corpus: which artists, decades, and chart positions are represented?
3. **Parse** chord annotation files into a tidy table ready for analysis
4. **Analyse** the harmonic vocabulary: what are the most common chords in pop/rock?
5. **Discover** chord progressions: which transitions between chords appear most often?
6. **Track** harmonic change over time: did vocabulary, chord quality usage, or entropy shift across three decades?


---
## Part 1: Setup and Data Loading

We use the **mirdata** library to download and manage the McGill Billboard dataset. `mirdata` is a Python library that standardises access to common Music Information Retrieval datasets — it handles downloading, file management, and provides a clean API to access annotations and metadata.

The dataset contains two main components:

| Component | Contents |
|-----------|----------|
| **Index** | Metadata for every sampled chart slot (artist, title, chart date, peak rank, weeks on chart) |
| **LAB files** | Chord annotations — one file per song with start time, end time, and chord label |


In [ ]:
%pip install pandas numpy matplotlib seaborn scipy mirdata --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mirdata

np.random.seed(42)

BG = "#F5F1E0"
PALETTE = ["#C00000", "#FFC000", "#4A90D9", "#7F3FBF", "#3A3A3A",
           "#E07A5F", "#AFCDCA", "#8B7355", "#2D936C", "#D4A5A5"]
plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor":   BG,
    "axes.edgecolor":   "#888888",
    "axes.labelcolor":  "#222222",
    "font.size":        12,
    "axes.titlesize":   14,
    "axes.titleweight": "bold",
    "grid.color":       "#CCCCCC",
    "grid.linestyle":   "--",
    "grid.alpha":       0.5,
})

print("Libraries loaded.")

In [ ]:
billboard = mirdata.initialize("billboard")
billboard.download()
print(f"Dataset ready. Number of tracks: {len(billboard.track_ids)}")

### Building the metadata table

`mirdata` gives us access to track objects with metadata and annotations. Let's build a pandas DataFrame from the metadata so we can explore the corpus.


In [ ]:
rows = []
for tid in billboard.track_ids:
    track = billboard.track(tid)
    rows.append({
        "track_id":       tid,
        "title":          track.title,
        "artist":         track.artist,
        "chart_date":     track.chart_date,
        "peak_rank":      track.peak_rank,
        "weeks_on_chart": track.weeks_on_chart,
    })

songs = pd.DataFrame(rows)
print(f"Songs in corpus: {len(songs)}")
songs.head()

Let's also peek at one track's chord annotations to see what we are working with.


In [ ]:
sample_track = billboard.track(billboard.track_ids[0])
print(f"Song: {sample_track.title} — {sample_track.artist}\n")

chords = sample_track.chords_full
print("First 10 chord events:")
for i in range(min(10, len(chords.labels))):
    start = chords.intervals[i, 0]
    end   = chords.intervals[i, 1]
    label = chords.labels[i]
    print(f"  {start:8.2f}s — {end:8.2f}s   {label}")

Each chord event has three pieces of information:

| Field | Meaning |
|-------|---------|
| **Start time** | When the chord begins (seconds) |
| **End time** | When the chord ends (seconds) |
| **Chord label** | e.g. `C:maj` (C major), `A:min` (A minor), `G:7` (G seventh chord), `N` (no chord / silence) |

The chord labels follow the Harte notation standard (Harte et al., 2005): `Root:Quality`. We will parse all of these into a single table in Part 3.


---
## Part 2: Exploring the Corpus Metadata

> **Research question: What does the Billboard Hot 100 sample look like?**

Before diving into chords, let's understand what is in this corpus. Who are the artists? Which decades are represented? How do chart positions relate to longevity?


In [ ]:
songs["year"] = pd.to_datetime(songs["chart_date"]).dt.year
songs["decade"] = (songs["year"] // 10) * 10

songs[["title", "artist", "year", "decade", "peak_rank", "weeks_on_chart"]].head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
songs["year"].hist(bins=range(1958, 1993), ax=ax, color=PALETTE[2], edgecolor="white")
ax.set_xlabel("Year")
ax.set_ylabel("Number of songs")
ax.set_title("Songs in the corpus by year")
plt.tight_layout()
plt.show()

In [ ]:
top_artists = songs["artist"].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 5))
top_artists.plot.barh(ax=ax, color=PALETTE[0])
ax.invert_yaxis()
ax.set_xlabel("Number of songs in corpus")
ax.set_title("Most represented artists")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(songs["peak_rank"], songs["weeks_on_chart"],
           alpha=0.4, s=20, color=PALETTE[3])
ax.set_xlabel("Peak rank (1 = highest)")
ax.set_ylabel("Weeks on chart")
ax.set_title("Chart performance: peak rank vs. longevity")
ax.invert_xaxis()
plt.tight_layout()
plt.show()

The scatter plot shows an interesting pattern: songs that reach higher peak positions (closer to #1) tend to spend more weeks on the chart, but there is substantial variation. Some songs peak high but drop off quickly, while others hover in the lower ranks for many weeks.


<div style="border: 2px solid #4A90D9; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #f0f7ff;">
<b>✏️ Task 1 — Describe the corpus</b><br><br>
Answer the following in a sentence or two each:<br>
1. Which decade has the <b>most</b> songs in the corpus?<br>
2. Who is the single most represented artist, and how many songs do they have?<br>
3. What is the median <code>peak_rank</code> in the corpus? (Hint: <code>songs["peak_rank"].median()</code>)<br>
</div>

---
## Part 3: Parsing Chord Annotations into a Table

> **Research question: How do we turn music annotations into data we can analyse?**

Each song has timed chord labels accessible via `mirdata`. We need to:
1. Read every track's chord data and combine them into **one big table**.
2. **Merge** with the metadata so we know each chord's song title, artist, and year.
3. **Extract** useful components from each chord label — the **root** pitch class and the **base quality**.

### Harte chord notation

The dataset uses **Harte notation** (Harte et al., 2005), a text-based format for chord symbols. The general structure is:

`Root:Quality(Extensions)/Bass` — for example `G:7(b9)/5`

- **Root** — pitch class: `C`, `D`, `Eb`, `F#`, etc.
- **Quality** — chord type: `maj`, `min`, `7`, `dim7`, `sus4`, etc.
- **(Extensions)** — optional alterations: `(b9)`, `(#11)`, etc.
- **/Bass** — optional bass note if different from the root

The special label **N** means "no chord" (silence or noise).


In [ ]:
all_chords = []
skipped = 0

for tid in billboard.track_ids:
    track = billboard.track(tid)
    try:
        chords = track.chords_full
    except (ValueError, Exception):
        skipped += 1
        continue
    if chords is None:
        skipped += 1
        continue
    for i in range(len(chords.labels)):
        all_chords.append({
            "track_id": tid,
            "start":    chords.intervals[i, 0],
            "end":      chords.intervals[i, 1],
            "chord":    chords.labels[i],
        })

chords_df = pd.DataFrame(all_chords)
chords_df["duration"] = chords_df["end"] - chords_df["start"]

print(f"Parsed {len(chords_df):,} chord events from {chords_df['track_id'].nunique()} songs")
if skipped:
    print(f"  ({skipped} tracks skipped due to missing or malformed data)")
chords_df.head()

In [ ]:
chords_df = chords_df.merge(songs, on="track_id", how="left")
print(f"Table shape after merge: {chords_df.shape}")
chords_df.head()

### Extracting root and base quality

Each Harte chord label has the form `Root:Quality(Extensions)/Bass`, e.g. `G:7(b9)/5`. We extract two useful columns:

- **root** — the pitch class of the chord (`C`, `F#`, `Bb`, …)
- **base quality** — the named quality *without* parenthetical extensions, e.g. `min7(11)` → `min7`, `7(b9)` → `7`, `maj(9)` → `maj`

We also do two important clean-up steps:
1. Remove `N` (no chord / silence) labels since they represent the absence of harmony.
2. **Collapse consecutive repetitions** — the raw annotations often repeat the same chord across multiple beat-level segments. For harmonic analysis we care about *chord changes* (compositional choices), not about how long each chord is sustained. That is a question of *harmonic rhythm*, which is a separate topic.


In [ ]:
def extract_root(label):
    """Return the root pitch class of a Harte chord label, or 'N'."""
    if label == "N" or ":" not in label:
        return label
    return label.split(":")[0]

def extract_base_quality(label):
    """Return the base quality (without extensions) of a Harte chord label.

    Examples:
        'C:maj7'       -> 'maj7'
        'G:7(b9)/5'    -> '7'
        'A:min'        -> 'min'
        'F:sus4(b7,9)' -> 'sus4'
    """
    if label == "N" or ":" not in label:
        return label
    quality = label.split(":")[1].split("/")[0]  # strip bass note
    return quality.split("(")[0]                  # strip extensions

chords_df["root"] = chords_df["chord"].apply(extract_root)
chords_df["base_quality"] = chords_df["chord"].apply(extract_base_quality)

print("Sample extractions:")
samples = chords_df[["chord", "root", "base_quality"]].drop_duplicates().head(15)
for _, row in samples.iterrows():
    print(f"  {row['chord']:20s} -> root={row['root']:4s}  quality={row['base_quality']}")

In [ ]:
n_before = len(chords_df)
harmony = chords_df[chords_df["chord"] != "N"].copy()
n_after_N = len(harmony)

# Collapse consecutive repetitions within each song
harmony = harmony[harmony["chord"] != harmony.groupby("track_id")["chord"].shift()].copy()
n_after_dedup = len(harmony)

print(f"Removed {n_before - n_after_N:,} 'N' (no-chord) segments")
print(f"Collapsed {n_after_N - n_after_dedup:,} consecutive repetitions")
print(f"Remaining chord changes: {n_after_dedup:,}")

<div style="border: 2px solid #4A90D9; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #f0f7ff;">
<b>✏️ Task 2 — Get to know the chord data</b><br><br>
Using the <code>harmony</code> DataFrame, answer:<br>
1. How many <b>unique chord labels</b> are there in total? (Hint: <code>harmony["chord"].nunique()</code>)<br>
2. Which <b>root pitch class</b> appears most often? What does this tell us about preferred keys in pop music? (Hint: <code>harmony["root"].value_counts()</code>)<br>
3. What is the <b>average number of chord changes per song</b>? Which song has the most? (Hint: <code>harmony.groupby("track_id").size()</code>)<br>
</div>

---
## Part 4: The Harmonic Vocabulary of Pop Music

> **Research question: What are the most common chords in pop/rock music?**

A widespread intuition is that pop songs are harmonically simple — relying on just a few chords. But how small is that vocabulary really, and which chords and chord qualities dominate? Let's look at the data.


In [ ]:
chord_counts = harmony["chord"].value_counts()
print(f"Total unique chords: {len(chord_counts)}\n")
print("Top 20 chords (by number of chord changes):")
chord_counts.head(20)

In [ ]:
top_n = 20
top_chords = chord_counts.head(top_n)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(range(top_n), top_chords.values, color=PALETTE[0], edgecolor="white")
ax.set_xticks(range(top_n))
ax.set_xticklabels(top_chords.index, rotation=45, ha="right", fontsize=11)
ax.set_ylabel("Occurrences")
ax.set_title(f"Top {top_n} most common chords (by chord changes)")
plt.tight_layout()
plt.show()

The distribution is strikingly **uneven**: a handful of major and minor chords account for the vast majority of all chord changes. This is consistent with the idea that pop music relies on a small, shared harmonic vocabulary.


In [ ]:
quality_counts = harmony["base_quality"].value_counts()

fig, ax = plt.subplots(figsize=(12, 5))
top_q = quality_counts.head(15)
bars = ax.bar(range(len(top_q)), top_q.values, color=PALETTE[1], edgecolor="white")
ax.set_xticks(range(len(top_q)))
ax.set_xticklabels(top_q.index, rotation=45, ha="right", fontsize=11)
ax.set_ylabel("Occurrences")
ax.set_title("Chord quality distribution (top 15 base qualities)")
plt.tight_layout()
plt.show()

print(f"\nTop 5 qualities cover {quality_counts.head(5).sum() / quality_counts.sum():.1%} of all events")

Major triads (`maj`) dominate by a wide margin, followed by minor triads (`min`) and seventh chords (`7`). Notice how the distribution drops off steeply — the top few qualities account for the vast majority of chord changes, while more colourful sonorities (augmented, diminished, suspended) are comparatively rare.


<div style="border: 2px solid #4A90D9; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #f0f7ff;">
<b>✏️ Task 3 — Cumulative coverage</b><br><br>
What <b>fraction</b> of all chord changes are accounted for by just the <b>top 5</b> chords?<br>
What about the <b>top 10</b>?<br><br>
Hint: use <code>chord_counts.head(5).sum() / chord_counts.sum()</code><br><br>
Bonus: do the same for <code>quality_counts</code>. How many base qualities do you need to cover 90% of the data?
</div>

---
## Part 5: Chord Progressions and Harmonic Change Over Time

> **Research question: Which chord transitions are most common, and did harmonic language change between 1958 and 1991?**

Individual chords are only half the story. The way chords **follow each other** — chord progressions — is what gives music its sense of movement and direction. A progression like V → I (dominant to tonic) creates a strong feeling of resolution; IV → V → I is the backbone of countless songs.

We start with **chord bigrams** (pairs of consecutive chords) and then investigate how the harmonic vocabulary evolved over three decades.


In [ ]:
def get_bigrams(group, column="chord"):
    """Return bigrams (consecutive pairs) from a column for one song."""
    vals = group[column].values
    return list(zip(vals[:-1], vals[1:]))

bigram_list = []
for _, group in harmony.groupby("track_id"):
    bigram_list.extend(get_bigrams(group, "chord"))

bigrams = pd.DataFrame(bigram_list, columns=["from_chord", "to_chord"])
bigrams = bigrams[bigrams["from_chord"] != bigrams["to_chord"]]

bigrams["progression"] = bigrams["from_chord"] + " → " + bigrams["to_chord"]
print(f"Total chord transitions (excl. self-repetitions): {len(bigrams):,}")

In [ ]:
top_prog = bigrams["progression"].value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 6))
top_prog.plot.barh(ax=ax, color=PALETTE[2])
ax.invert_yaxis()
ax.set_xlabel("Occurrences")
ax.set_title("Top 20 most common chord progressions (bigrams)")
plt.tight_layout()
plt.show()

### Reading the results

The most frequent transitions reveal the "grammar" of pop harmony:
- **Tonic ↔ subdominant and dominant** movements dominate (e.g. G:maj → D:maj, C:maj → F:maj, C:maj → G:maj) — these are the I → IV and I → V progressions that form the backbone of tonal harmony.
- Nearly all top bigrams are **reciprocal**: if A → B is common, so is B → A.
- The top 20 consists entirely of **major → major** transitions, because major chords are so much more frequent in absolute terms. Progressions involving minor or seventh chords are present but buried further down the ranking.


### Harmony over the decades

> **Research question: Did the harmonic language of Billboard hits change between 1958 and 1991?**

The corpus spans over three decades of pop music. We can ask whether harmonic conventions evolved — did songs become simpler or richer? Did the mix of chord qualities shift? We explore this through three lenses: vocabulary size, quality proportions, and entropy.

#### 1. Harmonic vocabulary size

How many distinct chords does a typical Billboard hit use? Has this changed over time?


In [ ]:
vocab = harmony.groupby(["track_id", "year", "decade"])["chord"].nunique().reset_index(name="n_unique")
vocab["decade_label"] = vocab["decade"].astype(int).astype(str) + "s"

fig, ax = plt.subplots(figsize=(10, 5))
decade_order = sorted(vocab["decade_label"].unique())
sns.boxplot(data=vocab, x="decade_label", y="n_unique", hue="decade_label",
            ax=ax, order=decade_order, palette="YlOrRd", legend=False, fliersize=3)
ax.set_xlabel("Decade")
ax.set_ylabel("Unique chords per song")
ax.set_title("Harmonic vocabulary size by decade")
plt.tight_layout()
plt.show()

print(f"Median unique chords per song by decade:")
for dl in decade_order:
    med = vocab[vocab["decade_label"] == dl]["n_unique"].median()
    print(f"  {dl}: {med:.0f}")

The boxplot gives a clean decade-level summary, but it hides year-to-year variation. The scatter plot below shows every song individually, plotted by its chart year, so we can spot trends, outliers, or abrupt shifts that the decade grouping might smooth over.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

decade_colors = {d: c for d, c in zip(
    sorted(vocab["decade"].unique()),
    sns.color_palette("YlOrRd", n_colors=len(vocab["decade"].unique()))
)}
colors = vocab["decade"].map(decade_colors)

ax.scatter(vocab["year"], vocab["n_unique"], alpha=0.4, s=20, c=colors.tolist())

yearly_median = vocab.groupby("year")["n_unique"].median()
ax.plot(yearly_median.index, yearly_median.values, color="black", linewidth=2, label="Yearly median")

for d in sorted(vocab["decade"].unique()):
    ax.axvspan(d, d + 10, alpha=0.06, color=decade_colors[d])

ax.set_xlabel("Year")
ax.set_ylabel("Unique chords per song")
ax.set_title("Harmonic vocabulary size by year (coloured by decade)")
ax.legend()
plt.tight_layout()
plt.show()

#### 2. Chord quality proportions by decade

Has the balance between major, minor, seventh, and other chord types shifted over the decades?


In [ ]:
top_qualities = harmony["base_quality"].value_counts().head(6).index.tolist()

qual_decade = harmony.copy()
qual_decade["quality_group"] = qual_decade["base_quality"].where(
    qual_decade["base_quality"].isin(top_qualities), other="other"
)

props = qual_decade.groupby(["decade", "quality_group"]).size().unstack(fill_value=0)
props = props.div(props.sum(axis=1), axis=0)

quality_order = top_qualities + ["other"]
props = props[[q for q in quality_order if q in props.columns]]

fig, ax = plt.subplots(figsize=(10, 6))
props.plot.bar(stacked=True, ax=ax, colormap="Set2", edgecolor="white", width=0.7)
ax.set_xlabel("Decade")
ax.set_ylabel("Proportion of chord changes")
ax.set_title("Chord quality proportions by decade")
ax.legend(title="Quality", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.set_xticklabels([str(int(x)) + "s" for x in props.index], rotation=0)
plt.tight_layout()
plt.show()

#### 3. Chord entropy over time

A more rigorous way to measure harmonic diversity is **Shannon entropy**:

$$H = -\sum_{i} p_i \log_2 p_i$$

where $p_i$ is the proportion of chord $i$ in a song. A song that uses only two chords equally has low entropy; a song that spreads its harmony across many chords has high entropy. This captures not just how many chords a song uses, but how evenly it distributes them.

In [ ]:
from scipy.stats import entropy as shannon_entropy

def song_entropy(group):
    counts = group["chord"].value_counts()
    return shannon_entropy(counts, base=2)

ent = (harmony.groupby(["track_id", "year", "decade"])["chord"]
       .apply(lambda g: shannon_entropy(g.value_counts(), base=2))
       .reset_index(name="entropy"))
ent["decade_label"] = ent["decade"].astype(int).astype(str) + "s"

fig, ax = plt.subplots(figsize=(10, 5))
decade_order = sorted(ent["decade_label"].unique())
sns.boxplot(data=ent, x="decade_label", y="entropy", hue="decade_label",
            ax=ax, order=decade_order, palette="YlOrRd", legend=False, fliersize=3)
ax.set_xlabel("Decade")
ax.set_ylabel("Shannon entropy (bits)")
ax.set_title("Chord entropy by decade")
plt.tight_layout()
plt.show()

print(f"Median entropy by decade:")
for dl in decade_order:
    med = ent[ent["decade_label"] == dl]["entropy"].median()
    print(f"  {dl}: {med:.2f} bits")

<div style="border: 2px solid #4A90D9; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #f0f7ff;">
<b>✏️ Task 4 — Interpret the diachronic trends</b><br><br>
Look at the three plots above and consider:<br>
1. Does harmonic <b>vocabulary size</b> increase, decrease, or stay stable over the decades?<br>
2. Do the <b>quality proportions</b> shift noticeably? Which qualities gain or lose ground?<br>
3. What does the <b>entropy</b> trend tell us — is pop harmony becoming more or less predictable?<br>
4. Miles et al. (2021) proposed the "Inflationary Surprise Hypothesis" — that top-charting songs tend to be more harmonically surprising and that this effect grows over time. Do your observations support or challenge this?<br>
</div>

<div style="border: 2px solid #4A90D9; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #f0f7ff;">
<b>✏️ Task 5 — Explore a specific artist's harmony</b><br><br>
Pick an artist who appears in the corpus (check <code>songs["artist"].value_counts()</code>) and repeat the bigram analysis <b>for that artist only</b>.<br><br>
1. Filter <code>harmony</code> to only that artist's songs.<br>
2. Compute bigrams and show the top 10 progressions.<br>
3. How do they compare to the corpus-wide results?<br><br>
<em>Hint: </em><code>artist_chords = harmony[harmony["artist"] == "Your Artist"]</code>
</div>

---
## Summary

In this session we:

1. **Loaded** the McGill Billboard dataset — chord-annotated songs from the Billboard Hot 100 (1958–1991) — using the `mirdata` library.
2. **Explored** the corpus metadata and discovered which artists and decades are most represented.
3. **Parsed** chord annotations (in Harte notation) into a structured table, extracted root pitch classes and base qualities, and collapsed consecutive repetitions to count chord changes.
4. **Found** that pop/rock harmony is built on a surprisingly small vocabulary — a handful of major and minor chords account for the vast majority of all chord changes, and the quality distribution drops off steeply after the first few types.
5. **Mapped** chord progressions (bigrams) and identified the most common harmonic transitions in the corpus.
6. **Tracked** harmonic change over three decades — vocabulary size, quality proportions, and Shannon entropy — connecting our findings to the "harmonic surprise" literature.

### Connection to current research

This kind of corpus-based chord analysis is an active area in computational musicology. For example:

- **Miles et al. (2021)** used the McGill Billboard corpus to show that *harmonic surprise* (measured with information theory) has increased over time in preferred songs — the "Inflationary Surprise Hypothesis." Top-charting songs tend to be more harmonically surprising than lower-charting ones, and this effect grows stronger over the decades.
- The **DataCamp / Kaggle** "Wrangling and Visualizing Musical Data" project uses the same dataset to compare chord usage in piano-driven vs. guitar-driven artists.

### References

- Burgoyne, J. A., Wild, J., & Fujinaga, I. (2011). An expert ground truth set for audio chord recognition and music analysis. *Proceedings of ISMIR*, 633–638.
- Miles, S. A., Rosen, D. S., Barry, S., Grunberg, D., & Grzywacz, N. (2021). What to expect when the unexpected becomes expected: Harmonic surprise and preference over time in popular music. *Frontiers in Human Neuroscience*, 15, 578644.
- Harte, C. A., Sandler, M. B., Abdallah, S. A., & Gómez, E. (2005). Symbolic representation of musical chords: A proposed syntax for text annotations. *Proceedings of ISMIR*, 66–71.
